# Notebook 05 Part 2: Lazy Evaluation and Lineage
```

## 1. What Is Lazy Evaluation?

**Lazy evaluation** means Spark does not execute every operation immediately.

When we write:

```python
df_filtered = df.filter(df.amount > 500)
```

Spark only records the instruction:

```text
Filter rows where amount is greater than 500
```

It does not immediately process the complete data.

Execution starts when we call an **action**, such as:

```python
df_filtered.show()
```

## 2. Simple Analogy

Imagine giving a chef these instructions:

```text
1. Take vegetables
2. Wash them
3. Cut them
4. Cook them
5. Serve the food
```

The first four steps describe the work.
The chef completes the full process when food must be served.

In Spark:

```text
Transformations build the plan.
Actions trigger the execution.
```

## 3. Transformation vs Action

| Type | Meaning | Examples |
|---|---|---|
| Transformation | Creates a new RDD or DataFrame and adds a step to the plan | `filter()`, `select()`, `map()`, `withColumn()` |
| Action | Requests a result and triggers execution | `show()`, `count()`, `collect()`, `take()` |

Important rule:

```text
Transformation = Spark remembers the work
Action = Spark performs the work
```

## 4. Why Does Spark Use Lazy Evaluation?

Lazy evaluation allows Spark to inspect the complete sequence of operations before execution.

This helps Spark:

```text
1. Build an execution plan
2. Combine related operations
3. Remove unnecessary work
4. Read only required columns
5. Apply filters early
6. Execute the plan efficiently
```

## 5. Windows Local Setup

Run this cell before creating the `SparkSession`.

In [12]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

## 6. Create SparkSession

In [13]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lazy_Evaluation_and_Lineage") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

## 7. Create a Sample DataFrame

In [14]:
data = [
    (1, "Amit", "Delhi", 500),
    (2, "Priya", "Mumbai", 800),
    (3, "Rahul", "Delhi", 300),
    (4, "Sneha", "Pune", 900),
    (5, "Karan", "Mumbai", 600)
]

columns = ["order_id", "customer_name", "city", "amount"]

df = spark.createDataFrame(data, columns)
df.show()

+--------+-------------+------+------+
|order_id|customer_name|  city|amount|
+--------+-------------+------+------+
|       1|         Amit| Delhi|   500|
|       2|        Priya|Mumbai|   800|
|       3|        Rahul| Delhi|   300|
|       4|        Sneha|  Pune|   900|
|       5|        Karan|Mumbai|   600|
+--------+-------------+------+------+



## 8. Build a Transformation Plan

The following operations are transformations:

```python
filter()
select()
withColumn()
```

Spark records these operations, but it does not need to produce the final result yet.

In [15]:
from pyspark.sql.functions import col

df_filtered = df.filter(col("amount") > 500)

df_selected = df_filtered.select(
    "customer_name",
    "city",
    "amount"
)

df_final = df_selected.withColumn(
    "discount_amount",
    col("amount") * 0.10
)

print("Transformations have been defined.")

Transformations have been defined.


At this point, Spark has built a plan similar to:

```text
Read Data
   ↓
Filter amount > 500
   ↓
Select required columns
   ↓
Create discount_amount
```

## 9. Trigger Execution with an Action

`show()` is an action.

It asks Spark to execute the plan and display the result.

In [16]:
df_final.show()

+-------------+------+------+---------------+
|customer_name|  city|amount|discount_amount|
+-------------+------+------+---------------+
|        Priya|Mumbai|   800|           80.0|
|        Sneha|  Pune|   900|           90.0|
|        Karan|Mumbai|   600|           60.0|
+-------------+------+------+---------------+



## 10. Another Action: `count()`

Each action requests a result.

Therefore, Spark may execute the required plan again unless the result has been cached.

In [17]:
row_count = df_final.count()
print("Number of rows:", row_count)

Number of rows: 3


## 11. What Is Lineage?

**Lineage** is the history of transformations used to create an RDD or DataFrame.

Example:

```text
Original data
   ↓ filter()
Filtered data
   ↓ select()
Selected data
   ↓ withColumn()
Final data
```

Spark remembers:

```text
Where the data came from
Which transformations were applied
How the final result can be recreated
```

## 12. Why Is Lineage Important?

Suppose one partition is lost because a worker fails.

Spark can use lineage to recreate the lost partition.

```text
Lost partition
      ↓
Check lineage
      ↓
Repeat the required transformations
      ↓
Recreate the partition
```

This is called **fault tolerance**.

## 13. RDD Lineage Example

RDD lineage can be inspected using:

```python
rdd.toDebugString()
```

In [18]:
sc = spark.sparkContext

numbers_rdd = sc.parallelize([1, 2, 3, 4, 5, 6], 2) # 2 means number of partition

even_rdd = numbers_rdd.filter(lambda x: x % 2 == 0)

squared_rdd = even_rdd.map(lambda x: x ** 2)

No action has been called yet.

Spark has only recorded:

```text
Create RDD
   ↓
Keep even numbers
   ↓
Square each number
```

In [19]:
print(squared_rdd.toDebugString().decode("utf-8"))

(2) PythonRDD[32] at RDD at PythonRDD.scala:58 []
 |  ParallelCollectionRDD[31] at readRDDFromFile at PythonRDD.scala:299 []


The lineage output shows how the final RDD depends on earlier RDDs.

Exact text may differ depending on the Spark version.

## 14. Trigger RDD Execution

`collect()` is an action.

In [20]:
squared_rdd.collect()

[4, 16, 36]

## 15. DataFrame Execution Plan

For DataFrames, we inspect the execution plan using:

```python
df.explain()
```

In [21]:
df_final.explain()

== Physical Plan ==
*(1) Project [customer_name#40, city#41, amount#42L, (cast(amount#42L as double) * 0.1) AS discount_amount#56]
+- *(1) Filter (isnotnull(amount#42L) AND (amount#42L > 500))
   +- *(1) Scan ExistingRDD[order_id#39L,customer_name#40,city#41,amount#42L]




Read the plan mainly from **bottom to top**.

You may see operations related to:

```text
Reading the original data
Filtering rows
Selecting columns
Creating the new column
```

## 16. Lazy Evaluation Flow

```text
Write transformation
        ↓
Spark records the instruction
        ↓
Write another transformation
        ↓
Spark adds it to the plan
        ↓
Call an action
        ↓
Spark optimizes and executes the plan
        ↓
Result is returned
```

## 17. Important Clarification

Creating a variable does not necessarily mean Spark has processed all data.

Example:

```python
df_filtered = df.filter(df.amount > 500)
```

This creates a new DataFrame object containing a plan.

Execution happens when an action requests a result:

```python
df_filtered.show()
df_filtered.count()
df_filtered.collect()
```

## 18. Common Transformations and Actions

### Transformations

```text
filter()
select()
withColumn()
drop()
distinct()
groupBy()
join()
map()
```

### Actions

```text
show()
count()
collect()
take()
first()
write()
```

Note:

```text
groupBy() starts building an aggregation plan.
The result is executed when an action is called.
```

## 19. Lazy Evaluation and Optimization Example

Suppose we write:

```python
result = df.select("customer_name", "amount") \
           .filter(col("amount") > 500)
```

Spark can examine both operations together.

It may optimize the plan by:

```text
Keeping only required columns
Applying the filter efficiently
Avoiding unnecessary processing
```

## 20. Lineage vs Execution Plan

| Concept | Simple Meaning |
|---|---|
| Lineage | History of how data was created |
| Execution plan | Plan Spark uses to perform the work |
| Lazy evaluation | Waiting until an action before executing |
| Fault tolerance | Recreating lost data from previous steps |

Easy relationship:

```text
Transformations create lineage and build a plan.
Actions trigger execution.
Lineage helps Spark recover lost data.
```

In [ ]:
spark.stop()